# 🚦 IntelliDrive AI: Model Training & Evaluation

This notebook covers the training process for the CNN Traffic Sign Classifier and the YOLO Object Detection integration.

In [1]:
import os
from ultralytics import YOLO
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

# Reinforcement Learning Libraries
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# Set paths (relative to notebook in experiments/ folder)
DATASET_PATH = '../dataset2/traffic_Data/DATA'
TEST_PATH = '../dataset2/traffic_Data/TEST'
LABELS_PATH = '../dataset2/labels.csv'
print(f"Using dataset at: {DATASET_PATH}")

Using dataset at: ../dataset2/traffic_Data/DATA


## 1. Data Analysis (Statistics)
Calculating mean, median, and mode for image distributions.

In [2]:
def get_stats(data_path, test_path):
    if not os.path.exists(data_path):
        print(f"Error: Path {data_path} does not exist.")
        return {}, []
    
    classes = os.listdir(data_path)
    print(f"Found {len(classes)} classes in DATA folder.")
    
    # Gather all image paths and labels for RL environment mapping
    all_images = []
    class_to_idx = {cls: idx for idx, cls in enumerate(sorted(classes))}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}
    
    total_train = 0
    for cls in classes:
        cls_dir = os.path.join(data_path, cls)
        imgs = [f for f in os.listdir(cls_dir) if f.endswith('.png') or f.endswith('.jpg')]
        total_train += len(imgs)
        for img in imgs:
            all_images.append((os.path.join(cls_dir, img), class_to_idx[cls]))
            
    total_test = len(os.listdir(test_path)) if os.path.exists(test_path) else 0
    
    print(f"Total training images: {total_train}")
    print(f"Total test images: {total_test}")
    
    label_map = {}
    if os.path.exists(LABELS_PATH):
        labels_df = pd.read_csv(LABELS_PATH)
        label_map = dict(zip(labels_df['ClassId'].astype(str), labels_df['Name']))
        print(f"Loaded {len(label_map)} class names from labels.csv")
        
    return label_map, all_images, class_to_idx, idx_to_class

label_map, train_images_list, class_to_idx, idx_to_class = get_stats(DATASET_PATH, TEST_PATH)

Found 58 classes in DATA folder.
Total training images: 4170
Total test images: 1994
Loaded 58 class names from labels.csv


## 2. YOLOv8 Training
Training YOLO with the custom dataset.

## 2. YOLOv8 Classification Training
YOLOv8 can also be used as a high-speed Image Classifier. We will train it directly on the `dataset2/traffic_Data/DATA` directories.

In [3]:
import os
from ultralytics import YOLO

# Build paths locally to notebook
DATASET_PATH_YOLO = os.path.abspath(DATASET_PATH)
print(f"YOLO Training on: {DATASET_PATH_YOLO}")

# Load a YOLOv8 classification model
yolo_model = YOLO('yolov8n-cls.pt')

# Train the model for image classification
# (Using 5 epochs for speed of demonstration)
results = yolo_model.train(
    data=DATASET_PATH_YOLO, 
    epochs=5, 
    imgsz=64, 
    batch=64
)
print("YOLOv8 Classification Training Complete!")

YOLO Training on: C:\Users\Arron\Documents\traffic\IntelliAIDrive\dataset2\traffic_Data\DATA


Ultralytics 8.4.21  Python-3.11.4 torch-2.10.0+cpu CPU (12th Gen Intel Core i7-12700H)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Arron\Documents\traffic\IntelliAIDrive\dataset2\traffic_Data\DATA, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrain

WARNING Dataset 'split=train' not found at C:\Users\Arron\Documents\traffic\IntelliAIDrive\dataset2\traffic_Data\DATA\train


Found 4170 images in subdirectories. Attempting to split...


...

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 7/7 1.4it/s 5.0s

                   all      0.776      0.925


Speed: 0.0ms preprocess, 0.4ms inference, 0.0ms loss, 0.0ms postprocess per image


Results saved to C:\Users\Arron\runs\classify\train


YOLOv8 Classification Training Complete!


In [4]:
# Visualize YOLO metrics
from IPython.display import Image, display

runs_dir = 'runs/classify'
latest_train_dir = None
if os.path.exists(runs_dir):
    train_dirs = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir) if d.startswith('train')]
    if train_dirs:
        latest_train_dir = max(train_dirs, key=os.path.getmtime)
        print(f"Using analytics from: {latest_train_dir}")

if latest_train_dir and os.path.exists(os.path.join(latest_train_dir, 'results.png')):
    print("Training Metrics:")
    display(Image(filename=os.path.join(latest_train_dir, 'results.png'), width=800))
if latest_train_dir and os.path.exists(os.path.join(latest_train_dir, 'confusion_matrix.png')):
    print("Confusion Matrix:")
    display(Image(filename=os.path.join(latest_train_dir, 'confusion_matrix.png'), width=800))


## 3. Reinforcement Learning (PPO) for Traffic Sign Classification
Formulating the image classification task as a Reinforcement Learning environment where the agent attempts to guess the class.

In [5]:
import cv2
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class TrafficSignEnv(gym.Env):
    """Custom Environment that follows gym interface for Image Classification"""
    metadata = {'render.modes': ['console']}

    def __init__(self, images_list, num_classes):
        super(TrafficSignEnv, self).__init__()
        self.images_list = images_list
        self.num_classes = num_classes
        self.current_step = 0
        self.max_steps = 1000 # Episodes span 1000 uniform samples
        
        # Action space: Predict one of the classes
        self.action_space = spaces.Discrete(self.num_classes)
        
        # Observation space: 64x64 RGB images as uint8
        self.observation_space = spaces.Box(low=0, high=255, shape=(64, 64, 3), dtype=np.uint8)
        self.current_image = None
        self.current_label = None
        
    def _load_random_image(self):
        img_path, label = random.choice(self.images_list)
        
        # Load and resize using OpenCV
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (64, 64))
        
        self.current_image = img
        self.current_label = label
        return self.current_image

    def step(self, action):
        # Check if the predicted class matches the actual label
        if action == self.current_label:
            reward = 1.0
        else:
            reward = -1.0
            
        self.current_step += 1
        done = self.current_step >= self.max_steps
        truncated = False
        
        # Get next state
        next_state = self._load_random_image()
        
        return next_state, reward, done, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        # Load a random image to start
        obs = self._load_random_image()
        return obs, {}

num_classes = len(class_to_idx)
env = TrafficSignEnv(train_images_list, num_classes)

# Verify the custom environment
check_env(env, warn=True)
print("TrafficSignEnv successfully created and verified.")

TrafficSignEnv successfully created and verified.


In [6]:
from stable_baselines3 import PPO

# Initialize the RL Agent with a CnnPolicy suited for images
print("Initializing PPO agent...")
model = PPO("CnnPolicy", env, verbose=1, learning_rate=0.0003, batch_size=64)

# Train the agent (Using 10,000 timesteps for demonstration, increase for better accuracy)
print("Training PPO agent...")
model.learn(total_timesteps=10000)

print("Agent trained successfully!")
# Save the trained agent
model.save("ppo_traffic_sign_classifier")

Initializing PPO agent...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
Training PPO agent...


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -971     |
| time/              |          |
|    fps             | 464      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -963        |
| time/                   |             |
|    fps                  | 296         |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.045915745 |
|    clip_fraction        | 0.425       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.03       |
|    explained_variance   | -0.00353    |
|    learning_rate        | 0.0003      |
|    loss                 | 2.93        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00567    |
|    value_loss           | 15.3        |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -960        |
| time/                   |             |
|    fps                  | 269         |
|    iterations           | 3           |
|    time_elapsed         | 22          |
|    total_timesteps      | 6144        |
| train/                  |             |
|    approx_kl            | 0.028913736 |
|    clip_fraction        | 0.397       |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.95       |
|    explained_variance   | -0.155      |
|    learning_rate        | 0.0003      |
|    loss                 | 4.33        |
|    n_updates            | 20          |
|    policy_gradient_loss | 0.0072      |
|    value_loss           | 16.6        |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 1e+03      |
|    ep_rew_mean          | -956       |
| time/                   |            |
|    fps                  | 242        |
|    iterations           | 4          |
|    time_elapsed         | 33         |
|    total_timesteps      | 8192       |
| train/                  |            |
|    approx_kl            | 0.30742943 |
|    clip_fraction        | 0.583      |
|    clip_range           | 0.2        |
|    entropy_loss         | -3.79      |
|    explained_variance   | -0.124     |
|    learning_rate        | 0.0003     |
|    loss                 | 16.3       |
|    n_updates            | 30         |
|    policy_gradient_loss | 0.0586     |
|    value_loss           | 22.5       |
----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 1e+03      |
|    ep_rew_mean          | -954       |
| time/                   |            |
|    fps                  | 230        |
|    iterations           | 5          |
|    time_elapsed         | 44         |
|    total_timesteps      | 10240      |
| train/                  |            |
|    approx_kl            | 0.58511543 |
|    clip_fraction        | 0.56       |
|    clip_range           | 0.2        |
|    entropy_loss         | -3.58      |
|    explained_variance   | -0.0981    |
|    learning_rate        | 0.0003     |
|    loss                 | 1.34       |
|    n_updates            | 40         |
|    policy_gradient_loss | 0.0534     |
|    value_loss           | 31.9       |
----------------------------------------


Agent trained successfully!


In [7]:
import os
from IPython.display import Image, display

# Find the most recent training run directory
runs_dir = 'runs/detect'
latest_train_dir = None

if os.path.exists(runs_dir):
    train_dirs = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir) if d.startswith('train')]
    if train_dirs:
        latest_train_dir = max(train_dirs, key=os.path.getmtime)
        print(f"Using analytics from: {latest_train_dir}")
else:
    print("Training run directory not found. Please run the training cell first.")

Using analytics from: runs/detect\train5


## 6. Test RL Agent on Dataset2 TEST Images & Metrics
We'll use our trained PPO agent to predict the class for images in the TEST folder and measure its F1-Score/Accuracy.
*Note: RL performs supervised classification by picking actions given environmental states (images).*

In [8]:
all_preds = []
all_labels = []

test_images = [f for f in os.listdir(TEST_PATH) if f.endswith('.png') or f.endswith('.jpg')]
print(f"Evaluating on {len(test_images)} test images...")

test_display_images = []
for i, img_name in enumerate(test_images):
    img_path = os.path.join(TEST_PATH, img_name)
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (64, 64))
    
    # Predict action using RL agent
    action, _state = model.predict(img_resized, deterministic=True)
    
    # Parse ground truth from filename
    try:
        class_str = str(int(img_name.split('_')[0]))
        true_label = class_to_idx.get(class_str, 0)
    except:
        true_label = 0
        
    all_preds.append(int(action))
    all_labels.append(true_label)
    
    # Save a few samples for display
    if i < 5:
        test_display_images.append((img_resized, int(action), true_label))

# Calculate metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"\nTest Accuracy: {acc*100:.2f}%")
print(f"Test F1 Score (weighted): {f1:.4f}")

# Visual predictions
plt.figure(figsize=(15, 5))
for i, (img, pred_idx, true_idx) in enumerate(test_display_images):
    pred_class_str = idx_to_class.get(pred_idx, str(pred_idx))
    true_class_str = idx_to_class.get(true_idx, str(true_idx))
    
    pred_name = label_map.get(pred_class_str, pred_class_str)
    true_name = label_map.get(true_class_str, true_class_str)
    
    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.title(f"T: {true_name}\nP: {pred_name}", fontsize=10)
    plt.axis('off')
    
plt.tight_layout()
plt.show()

Evaluating on 1994 test images...



Test Accuracy: 0.00%
Test F1 Score (weighted): 0.0000


<Figure size 1500x500 with 5 Axes>